# ARDY on Kaggle — T4 × 2 FINAL V2

针对 **Kaggle 2× NVIDIA T4（每张约 14.5 GiB）** 的最终稳定版。

- `cuda:0`：ARDY Core Horizon8
- `cuda:1`：LLM2Vec / Llama-3 8B，固定 bitsandbytes 4-bit NF4
- 8-bit OOM 自动回退 4-bit NF4
- 修复 LLM2Vec 在双 GPU 下忽略显式 `cuda:1`、误用 GPU0 的 multiprocessing 路径
- 使用 ARDY 官方 `CachedTextEncoder` 缓存 Prompt embedding
- Demo 默认 `--no-compile`
- ARDY 固定 commit：`693f74d13b3d04a0a22ce127ee79c929dd89756b`

第一次执行 Cell 0 会安装环境并自动重启 Kernel 一次；重启后从 Cell 0 重新运行。

In [ ]:
# Cell 0 — Bootstrap + T4 patch
from pathlib import Path
import os, signal, subprocess, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROOT = Path("/kaggle/working")
REPO = ROOT / "ardy"
ARDY_COMMIT = "693f74d13b3d04a0a22ce127ee79c929dd89756b"
MARKER = ROOT / ".ardy_t4x2_final_v1_ready"

def run(cmd, cwd=None):
    cmd = list(map(str, cmd))
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO.exists():
    run(["git", "clone", "https://github.com/nv-tlabs/ardy.git", REPO])
run(["git", "fetch", "origin", ARDY_COMMIT, "--depth", "1"], cwd=REPO)
run(["git", "checkout", "--detach", ARDY_COMMIT], cwd=REPO)

llm2vec_py = REPO / "ardy/model/llm2vec/llm2vec.py"
src = llm2vec_py.read_text()

old = 'if torch.cuda.device_count() <= 1:\n            # This branch also support mps devices\n            self.to(device)'
new = '''if torch.cuda.device_count() <= 1 or device is not None:
            # Kaggle T4x2 fix: respect explicitly requested device.
            is_quantized = bool(
                getattr(self.model, "is_loaded_in_4bit", False)
                or getattr(self.model, "is_loaded_in_8bit", False)
            )
            if not is_quantized:
                self.to(device)'''
if old in src:
    src = src.replace(old, new, 1)
elif "Kaggle T4x2 fix: respect explicitly requested device" not in src:
    raise RuntimeError("llm2vec encode patch target changed")

old = '        self.to(device)\n        features = self.tokenize([self.prepare_for_tokenization(sentence) for sentence in sentences_batch])'
new = '''        is_quantized = bool(
            getattr(self.model, "is_loaded_in_4bit", False)
            or getattr(self.model, "is_loaded_in_8bit", False)
        )
        if not is_quantized:
            self.to(device)
        features = self.tokenize([self.prepare_for_tokenization(sentence) for sentence in sentences_batch])'''
if old in src:
    src = src.replace(old, new, 1)
elif 'getattr(self.model, "is_loaded_in_8bit", False)' not in src:
    raise RuntimeError("llm2vec _encode patch target changed")

llm2vec_py.write_text(src)

quant_module = REPO / "ardy/model/kaggle_t4_text_encoder.py"
quant_module.write_text(r'''import gc
import os
import numpy as np
import torch
from transformers import BitsAndBytesConfig
from .llm2vec.llm2vec import LLM2Vec

BASE_REPO = "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp"
PEFT_REPO = "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised"

class QuantizedLLM2VecEncoder:
    def __init__(self, device="cuda:1", quantization="8bit", llm_dim=4096):
        self.llm_dim = llm_dim
        self._device = str(device)
        self.device = torch.device(device)
        self.dtype = torch.float16

        base, peft = BASE_REPO, PEFT_REPO
        root = os.environ.get("TEXT_ENCODERS_DIR")
        if root:
            lb = os.path.join(root, base)
            lp = os.path.join(root, peft)
            if os.path.isdir(lb): base = lb
            if os.path.isdir(lp): peft = lp

        if quantization == "8bit":
            qconfig = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization == "4bit":
            qconfig = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        else:
            raise ValueError(quantization)

        self.model = LLM2Vec.from_pretrained(
            base_model_name_or_path=base,
            peft_model_name_or_path=peft,
            torch_dtype=torch.float16,
            quantization_config=qconfig,
            device_map={"": self._device},
            cache_dir=os.environ.get("HUGGINGFACE_CACHE_DIR"),
            low_cpu_mem_usage=True,
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False

    def to(self, device=None, dtype=None):
        if device is not None and str(device) != self._device:
            raise RuntimeError(f"Quantized encoder pinned to {self._device}")
        return self

    def eval(self):
        self.model.eval()
        return self

    def get_device(self):
        return self._device

    def __call__(self, text):
        is_string = isinstance(text, str)
        texts = [text] if is_string else list(text)
        with torch.inference_mode():
            encoded = self.model.encode(
                texts,
                batch_size=1,
                show_progress_bar=False,
                convert_to_tensor=True,
                device=self._device,
            )
        encoded = encoded[:, None].to(device=self._device, dtype=torch.float16)
        lengths = np.ones(len(texts), dtype=int).tolist()
        return (encoded[0], lengths[0]) if is_string else (encoded, lengths)

def build_t4_text_encoder(device="cuda:1", preferred="4bit"):
    modes = [preferred] + (["4bit"] if preferred == "8bit" else [])
    last_error = None
    for mode in modes:
        try:
            print(f"Loading LLM2Vec on {device} with {mode}...", flush=True)
            enc = QuantizedLLM2VecEncoder(device=device, quantization=mode)
            os.environ["ARDY_TEXT_QUANT_ACTIVE"] = mode
            return enc
        except torch.OutOfMemoryError as exc:
            last_error = exc
            gc.collect()
            torch.cuda.empty_cache()
    raise last_error
''')

loading_py = REPO / "scripts/interactive_demo/loading.py"
loading = loading_py.read_text()
old = '        device = "cuda" if torch.cuda.is_available() else "cpu"\n        return load_text_encoder(mode="auto", device=device)'
new = '        from ardy.model.kaggle_t4_text_encoder import build_t4_text_encoder\n        return build_t4_text_encoder()'
if old in loading:
    loading = loading.replace(old, new, 1)
loading_py.write_text(loading)

if not MARKER.exists():
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
    run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "-e", ".[demo]", "bitsandbytes==0.50.2", "accelerate==1.14.0"
    ], cwd=REPO)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"])
    MARKER.write_text(ARDY_COMMIT + "\n")
    print("Installed; restarting Kaggle kernel once.")
    os.kill(os.getpid(), signal.SIGKILL)

print("Bootstrap ready:", REPO)

+ git clone https://github.com/nv-tlabs/ardy.git /kaggle/working/ardy


Cloning into '/kaggle/working/ardy'...


+ git fetch origin 693f74d13b3d04a0a22ce127ee79c929dd89756b --depth 1
+ git checkout --detach 693f74d13b3d04a0a22ce127ee79c929dd89756b


From https://github.com/nv-tlabs/ardy
 * branch            693f74d13b3d04a0a22ce127ee79c929dd89756b -> FETCH_HEAD


+ /usr/bin/python3 -m pip install --no-cache-dir --force-reinstall numpy==1.26.4


...

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
+ /usr/bin/python3 -m pip install --no-cache-dir -e .[demo] bitsandbytes==0.50.2 accelerate==1.14.0
Obtaining file:///kaggle/working/ardy
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/nv-tlabs/kimodo-viser.git (to revision 7c82ad8f8640bad9dff8ded5c5eee908eeb08f11) to /tmp/pip-install-k8msciwc/viser_efcf4edfd3d24e1ba9671e0a96ade311


  Running command git clone --filter=blob:none --quiet https://github.com/nv-tlabs/kimodo-viser.git /tmp/pip-install-k8msciwc/viser_efcf4edfd3d24e1ba9671e0a96ade311
  Running command git rev-parse -q --verify 'sha^7c82ad8f8640bad9dff8ded5c5eee908eeb08f11'
  Running command git fetch -q https://github.com/nv-tlabs/kimodo-viser.git 7c82ad8f8640bad9dff8ded5c5eee908eeb08f11


  Resolved https://github.com/nv-tlabs/kimodo-viser.git to commit 7c82ad8f8640bad9dff8ded5c5eee908eeb08f11
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 226.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 377.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 253.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 221.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 283.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 284.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 306.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.6.0 which is incompatible.


+ /usr/bin/python3 -m pip install --no-cache-dir --force-reinstall numpy==1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 253.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [1]:
# Cell 1 — 缓存、Token、双 T4 验证
from pathlib import Path
import os, sys, importlib.metadata as metadata

ROOT = Path("/kaggle/working")
REPO = ROOT / "ardy"
WORK_CACHE = ROOT / "ardy-cache"
PRELOADED = Path("/kaggle/input/ardy-preloaded")
WORK_CACHE.mkdir(parents=True, exist_ok=True)

HF_CACHE = WORK_CACHE / "huggingface"
HF_CACHE.mkdir(parents=True, exist_ok=True)

if (PRELOADED / "checkpoints").exists() and (PRELOADED / "text_encoders").exists():
    CHECKPOINTS_DIR = PRELOADED / "checkpoints"
    TEXT_ENCODERS_DIR = PRELOADED / "text_encoders"
    USING_PRELOADED = True
else:
    CHECKPOINTS_DIR = WORK_CACHE / "checkpoints"
    TEXT_ENCODERS_DIR = WORK_CACHE / "text_encoders"
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    TEXT_ENCODERS_DIR.mkdir(parents=True, exist_ok=True)
    USING_PRELOADED = False

EMBED_CACHE = WORK_CACHE / "text_embedding_cache"
EMBED_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HUGGINGFACE_CACHE_DIR"] = str(HF_CACHE)
os.environ["CHECKPOINTS_DIR"] = str(CHECKPOINTS_DIR)
os.environ["TEXT_ENCODERS_DIR"] = str(TEXT_ENCODERS_DIR)
os.environ["LOCAL_CACHE"] = "true"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "YES"
os.environ["TEXT_ENCODER_DEVICE"] = "cuda:1"
os.environ["ARDY_TEXT_QUANT"] = "4bit"

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded (not printed).")

import numpy as np, torch
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| PyTorch", torch.__version__, "| CUDA", torch.version.cuda)
print("Transformers", metadata.version("transformers"), "| bitsandbytes", metadata.version("bitsandbytes"))

assert np.__version__.startswith("1.")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "请在 Kaggle 选择 GPU T4 x2"

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}: {p.name} | {p.total_memory/1024**3:.2f} GiB | CC {p.major}.{p.minor}")

from ardy.model.kaggle_t4_text_encoder import build_t4_text_encoder
from ardy.model import load_model
print("Patched imports OK")

HF_TOKEN loaded (not printed).
Python 3.12.13 | NumPy 1.26.4 | PyTorch 2.10.0+cu128 | CUDA 12.8
Transformers 5.8.1 | bitsandbytes 0.50.2
cuda:0: Tesla T4 | 14.56 GiB | CC 7.5
cuda:1: Tesla T4 | 14.56 GiB | CC 7.5
Patched imports OK


In [2]:
# Cell 2 — 下载 ARDY Core8 + LLM2Vec
from pathlib import Path
from huggingface_hub import snapshot_download
import os

ARDY_MODEL_NAME = "ARDY-Core-RP-20FPS-Horizon8"
ARDY_LOCAL = CHECKPOINTS_DIR / ARDY_MODEL_NAME
TEXT_REPOS = [
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised",
]

def ensure_snapshot(repo_id, target, required_file=None):
    target = Path(target)
    ready = target.exists() and any(target.iterdir())
    if required_file:
        ready = ready and (target / required_file).exists()
    if ready:
        print("Ready:", repo_id)
        return
    if USING_PRELOADED:
        raise FileNotFoundError(target)
    snapshot_download(repo_id=repo_id, local_dir=str(target), token=os.environ.get("HF_TOKEN"))

ensure_snapshot("nvidia/" + ARDY_MODEL_NAME, ARDY_LOCAL, "config.yaml")
for repo_id in TEXT_REPOS:
    ensure_snapshot(repo_id, TEXT_ENCODERS_DIR / repo_id)
print("All model files ready.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

All model files ready.


In [3]:
# Cell 3 — T4x2 FINAL
# LLM2Vec 4-bit NF4 -> cuda:1

import gc
import os
import sys
import time
import torch

gc.collect()
torch.cuda.empty_cache()

# --------------------------------------------------
# GPU1 必须基本干净
# --------------------------------------------------

free0, total0 = torch.cuda.mem_get_info(0)
free1, total1 = torch.cuda.mem_get_info(1)

print(f"GPU0 free: {free0/1024**3:.2f} / {total0/1024**3:.2f} GiB")
print(f"GPU1 free: {free1/1024**3:.2f} / {total1/1024**3:.2f} GiB")

assert free1 > 13.5 * 1024**3, (
    "GPU1 不干净，请 Restart Kernel 后重新执行。"
)

# --------------------------------------------------
# 关键修复：
# 不允许 kaggle_t4_text_encoder.py 把本地 MNTP adapter
# 目录当成完整的模型目录
# --------------------------------------------------

_saved_text_encoders_dir = os.environ.pop(
    "TEXT_ENCODERS_DIR",
    None,
)

try:
    from ardy.model.kaggle_t4_text_encoder import (
        build_t4_text_encoder,
    )

    torch.cuda.reset_peak_memory_stats(1)

    t0 = time.perf_counter()

    # 直接 4-bit，不尝试 8-bit
    raw_text_encoder = build_t4_text_encoder(
        device="cuda:1",
        preferred="4bit",
    )

    torch.cuda.synchronize(1)

    print(
        f"LLM2Vec load time: "
        f"{time.perf_counter() - t0:.2f}s"
    )

finally:
    # 后续 ARDY 继续保留原环境变量
    if _saved_text_encoders_dir is not None:
        os.environ["TEXT_ENCODERS_DIR"] = (
            _saved_text_encoders_dir
        )

# --------------------------------------------------
# Prompt Cache
# --------------------------------------------------

scripts_dir = str(REPO / "scripts")

if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

from interactive_demo.embedding_cache import (
    CachedTextEncoder,
)

text_encoder = CachedTextEncoder(
    raw_text_encoder,
    cache_dir=str(EMBED_CACHE),
)

# --------------------------------------------------
# 真正测试 embedding
# --------------------------------------------------

prompt = "A person walks forward."

torch.cuda.synchronize(1)
t0 = time.perf_counter()

probe_embedding, probe_length = text_encoder(prompt)

torch.cuda.synchronize(1)

print()
print("===== LLM2Vec OK =====")
print("Quantization: 4-bit NF4")
print("Device:", raw_text_encoder.get_device())
print(
    "Embedding:",
    tuple(probe_embedding.shape),
    probe_embedding.dtype,
)
print(
    f"Encode time: "
    f"{time.perf_counter() - t0:.3f}s"
)

for gpu in (0, 1):
    free, total = torch.cuda.mem_get_info(gpu)

    print(
        f"GPU{gpu}: "
        f"allocated="
        f"{torch.cuda.memory_allocated(gpu)/1024**3:.2f} GiB | "
        f"free={free/1024**3:.2f} GiB"
    )

assert torch.cuda.memory_allocated(0) < 2 * 1024**3, (
    "LLM2Vec unexpectedly used GPU0"
)

# --------------------------------------------------
# 常用 Prompt Cache
# --------------------------------------------------

PROMPTS = [
    "A person walks forward.",
    "A person runs forward.",
    "A person turns left.",
    "A person turns right.",
    "A person jumps.",
]

for prompt in PROMPTS:
    t0 = time.perf_counter()

    emb, length = text_encoder(prompt)

    print(
        f"{time.perf_counter()-t0:7.3f}s | "
        f"{tuple(emb.shape)} | "
        f"{prompt}"
    )

print("Prompt cache ready:", EMBED_CACHE)

GPU0 free: 14.46 / 14.56 GiB
GPU1 free: 14.46 / 14.56 GiB
Loading LLM2Vec on cuda:1 with 4bit...


config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


adapter_config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  168MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

LLM2Vec load time: 129.64s

===== LLM2Vec OK =====
Quantization: 4-bit NF4
Device: cuda:1
Embedding: (1, 1, 4096) torch.float16
Encode time: 0.581s
GPU0: allocated=0.00 GiB | free=14.30 GiB
GPU1: allocated=4.50 GiB | free=7.25 GiB
  0.000s | (1, 1, 4096) | A person walks forward.
  0.134s | (1, 1, 4096) | A person runs forward.
  0.123s | (1, 1, 4096) | A person turns left.
  0.120s | (1, 1, 4096) | A person turns right.
  0.124s | (1, 1, 4096) | A person jumps.
Prompt cache ready: /kaggle/working/ardy-cache/text_embedding_cache


In [4]:
# Cell 4 — 修复 ardy.assets + ARDY Core8 -> cuda:0

import gc, sys, types, torch
from pathlib import Path
import ardy

MOTION_DEVICE = "cuda:0"
assets_dir = REPO / "ardy" / "assets"

# 强制修复 ardy.assets namespace 冲突（不重启 Kernel，不影响 GPU1 的 LLM2Vec）
assets_mod = types.ModuleType("ardy.assets")
assets_mod.__path__ = [str(assets_dir)]
assets_mod.ASSETS_ROOT = assets_dir
assets_mod.DEMO_ASSETS_ROOT = assets_dir / "demo"
assets_mod.DEMO_EXAMPLES_ROOT = assets_dir / "demo" / "examples"
assets_mod.SKELETONS_ROOT = assets_dir / "skeletons"
assets_mod.SOMA_ASSETS_ROOT = assets_dir / "SOMA"
assets_mod.skeleton_asset_path = lambda *parts: assets_dir.joinpath("skeletons", *parts)
assets_mod.demo_asset_path = lambda *parts: assets_dir.joinpath("demo", *parts)

sys.modules["ardy.assets"] = assets_mod
ardy.assets = assets_mod

from ardy.assets import skeleton_asset_path
print("assets fixed:", skeleton_asset_path("cskel27"))

# 清理 GPU0；GPU1 的 LLM2Vec 保留
gc.collect()
with torch.cuda.device(0): torch.cuda.empty_cache()

free0, total0 = torch.cuda.mem_get_info(0)
free1, total1 = torch.cuda.mem_get_info(1)
print(f"Before ARDY | GPU0 free={free0/1024**3:.2f} GiB | GPU1 free={free1/1024**3:.2f} GiB")

assert free0 > 12 * 1024**3, "GPU0 不干净，请检查是否有旧 ARDY 模型残留"

# 加载 ARDY
from ardy.model import load_model
model = load_model("core8", device=MOTION_DEVICE, text_encoder=text_encoder, checkpoints_dir=str(CHECKPOINTS_DIR))
model.eval()

print("\n===== ARDY OK =====")
print(f"FPS={model.motion_rep.fps} | Horizon={model.gen_horizon_len} | Frames/token={model.num_frames_per_token}")

for i in (0, 1):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU{i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GiB | reserved={torch.cuda.memory_reserved(i)/1024**3:.2f} GiB | free={free/1024**3:.2f} GiB")

assets fixed: /kaggle/working/ardy/ardy/assets/skeletons/cskel27
Before ARDY | GPU0 free=14.46 GiB | GPU1 free=7.28 GiB
sparsify_token_seq: False

===== ARDY OK =====
FPS=20 | Horizon=8 | Frames/token=4
GPU0: allocated=0.85 GiB | reserved=0.86 GiB | free=13.60 GiB
GPU1: allocated=4.50 GiB | reserved=7.15 GiB | free=7.28 GiB


In [5]:
# Cell 5 — Smoke Test
import time, torch
from ardy.motion_rep.tools import length_to_mask

fps = float(model.motion_rep.fps)
num_base_steps = int(model.diffusion.num_base_steps)
prompt = "A person walks forward."
duration = 2.0
num_frames = int(duration * fps)

pad_mask = length_to_mask(torch.tensor([num_frames], device=MOTION_DEVICE))
first_heading_angle = torch.zeros(1, device=MOTION_DEVICE)

torch.cuda.synchronize(0)
t0 = time.perf_counter()

with torch.inference_mode():
    motion = model(
        [prompt],
        num_frames,
        num_denoising_steps=num_base_steps,
        pad_mask=pad_mask,
        first_heading_angle=first_heading_angle,
        motion_mask=None,
        observed_motion=None,
        cfg_weight=(2.0, 2.0),
        crop_history_length=model.num_frames_per_token,
    )
    output = model.motion_rep.inverse(motion, is_normalized=True)

torch.cuda.synchronize(0)
dt = time.perf_counter() - t0

print(f"Generated {num_frames} frames | wall={dt:.3f}s | RTF={dt/duration:.3f}")
print("posed_joints:", output["posed_joints"].shape)

for i in (0, 1):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU{i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GiB | free={free/1024**3:.2f} GiB")

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Generated 40 frames | wall=1.345s | RTF=0.672
posed_joints: torch.Size([1, 40, 27, 3])
GPU0: allocated=0.86 GiB | free=13.55 GiB
GPU1: allocated=4.50 GiB | free=7.28 GiB


In [6]:
# Cell 6 — 保存 Smoke Test 输出
from pathlib import Path
import numpy as np

OUT = Path("/kaggle/working/ardy_outputs")
OUT.mkdir(parents=True, exist_ok=True)
posed_joints = output["posed_joints"].detach().float().cpu().numpy()
np.savez(OUT / "smoke_test.npz", posed_joints=posed_joints, fps=fps, prompt=prompt)
print("Saved:", OUT / "smoke_test.npz")

Saved: /kaggle/working/ardy_outputs/smoke_test.npz


## Interactive Demo

先释放 Notebook Kernel 里的模型，再启动 Demo 子进程，避免同一模型加载两遍造成 OOM。

In [10]:
# 先关闭旧的 Web Server
if "web_server" in globals():
    try:
        web_server.shutdown()
        web_server.server_close()
        print("Old web server closed.")
    except Exception as e:
        print("Close old server:", e)

if "web_thread" in globals():
    try:
        web_thread.join(timeout=2)
    except:
        pass

Old web server closed.


In [11]:
# Cell 7 — ARDY 自定义 3D 动作播放器
import json, time, threading, urllib.parse, uuid
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from pathlib import Path

import numpy as np
import torch
from ardy.motion_rep.tools import length_to_mask

assert "model" in globals(), "请先运行 Cell 4 加载 ARDY"
assert "text_encoder" in globals(), "请先运行 Cell 3 加载 LLM2Vec"

HOST, PORT = "0.0.0.0", 2333
OUT_DIR = Path("/kaggle/working/ardy_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
gpu_lock = threading.Lock()

HTML = r"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ARDY T4×2</title>

<style>
body{margin:0;background:#111;color:#eee;font-family:Arial,sans-serif}
main{max-width:1050px;margin:auto;padding:20px}
textarea,input,button{box-sizing:border-box;padding:10px;font-size:15px;border-radius:7px}
textarea,input{background:#222;color:white;border:1px solid #555}
textarea{width:100%;height:70px}
button{cursor:pointer;border:0;padding:10px 18px;font-weight:bold}
.row{display:flex;gap:10px;align-items:center;margin:10px 0}
#viewer{width:100%;height:600px;background:#080808;border-radius:10px;overflow:hidden}
#timeline{width:100%}
pre{background:#222;padding:12px;border-radius:8px;white-space:pre-wrap}
a{color:#7dc8ff}
</style>

<script type="importmap">
{
  "imports":{
    "three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js",
    "three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"
  }
}
</script>
</head>

<body>
<main>

<h2>ARDY — T4 × 2</h2>

<label>Prompt</label>
<textarea id="prompt">A person walks forward.</textarea>

<div class="row">
<label>Duration</label>
<input id="duration" type="number" value="2" min="0.5" max="10" step="0.5">
<span>seconds</span>
<button id="generate">Generate</button>
</div>

<div id="viewer"></div>

<div class="row">
<button id="play">▶ Play</button>
<span id="frameLabel">Frame 0 / 0</span>
</div>

<input id="timeline" type="range" min="0" max="0" value="0">

<pre id="result">Ready.</pre>

<script type="module">

import * as THREE from "three";
import {OrbitControls} from "three/addons/controls/OrbitControls.js";

const viewer=document.getElementById("viewer");

const scene=new THREE.Scene();
scene.background=new THREE.Color(0x080808);

const camera=new THREE.PerspectiveCamera(45,viewer.clientWidth/viewer.clientHeight,0.01,1000);
camera.position.set(3,2,5);

const renderer=new THREE.WebGLRenderer({antialias:true});
renderer.setPixelRatio(Math.min(devicePixelRatio,2));
renderer.setSize(viewer.clientWidth,viewer.clientHeight);
viewer.appendChild(renderer.domElement);

const controls=new OrbitControls(camera,renderer.domElement);
controls.enableDamping=true;

scene.add(new THREE.HemisphereLight(0xffffff,0x333333,3));

const grid=new THREE.GridHelper(10,20);
scene.add(grid);

const axes=new THREE.AxesHelper(1);
scene.add(axes);

let frames=[];
let fps=20;
let current=0;
let playing=false;
let lastTime=performance.now();

let joints=[];
let bones=null;
let edges=[];


/* 根据第一帧做最小生成树，自动得到大致骨架连线 */
function buildMST(points){

    const n=points.length;
    if(n<2)return [];

    const used=new Array(n).fill(false);
    const dist=new Array(n).fill(Infinity);
    const parent=new Array(n).fill(-1);

    dist[0]=0;

    for(let step=0;step<n;step++){

        let u=-1,best=Infinity;

        for(let i=0;i<n;i++){
            if(!used[i] && dist[i]<best){
                best=dist[i];
                u=i;
            }
        }

        if(u<0)break;
        used[u]=true;

        for(let v=0;v<n;v++){

            if(used[v])continue;

            const dx=points[u][0]-points[v][0];
            const dy=points[u][1]-points[v][1];
            const dz=points[u][2]-points[v][2];

            const d=dx*dx+dy*dy+dz*dz;

            if(d<dist[v]){
                dist[v]=d;
                parent[v]=u;
            }
        }
    }

    const result=[];

    for(let i=1;i<n;i++){
        if(parent[i]>=0)result.push([parent[i],i]);
    }

    return result;
}


function createSkeleton(count){

    for(const j of joints)scene.remove(j);
    joints=[];

    if(bones){
        scene.remove(bones);
        bones.geometry.dispose();
    }

    const geo=new THREE.SphereGeometry(0.035,12,12);
    const mat=new THREE.MeshStandardMaterial({color:0x55bbff});

    for(let i=0;i<count;i++){
        const mesh=new THREE.Mesh(geo,mat);
        scene.add(mesh);
        joints.push(mesh);
    }

    const linePositions=new Float32Array(edges.length*6);
    const lineGeo=new THREE.BufferGeometry();
    lineGeo.setAttribute("position",new THREE.BufferAttribute(linePositions,3));

    bones=new THREE.LineSegments(
        lineGeo,
        new THREE.LineBasicMaterial({color:0xffffff})
    );

    scene.add(bones);
}


function updateFrame(index){

    if(!frames.length)return;

    current=Math.max(0,Math.min(index,frames.length-1));

    const frame=frames[current];

    for(let i=0;i<frame.length;i++){
        const p=frame[i];

        /* ARDY XYZ -> Three XYZ */
        joints[i].position.set(p[0],p[1],p[2]);
    }

    const pos=bones.geometry.attributes.position.array;

    let k=0;

    for(const [a,b] of edges){

        const pa=frame[a];
        const pb=frame[b];

        pos[k++]=pa[0];
        pos[k++]=pa[1];
        pos[k++]=pa[2];

        pos[k++]=pb[0];
        pos[k++]=pb[1];
        pos[k++]=pb[2];
    }

    bones.geometry.attributes.position.needsUpdate=true;

    document.getElementById("timeline").value=current;
    document.getElementById("frameLabel").textContent=`Frame ${current+1} / ${frames.length}`;
}


function fitCamera(){

    if(!frames.length)return;

    const box=new THREE.Box3();

    for(const frame of frames){
        for(const p of frame){
            box.expandByPoint(new THREE.Vector3(p[0],p[1],p[2]));
        }
    }

    const center=new THREE.Vector3();
    const size=new THREE.Vector3();

    box.getCenter(center);
    box.getSize(size);

    const span=Math.max(size.x,size.y,size.z,1);

    controls.target.copy(center);

    camera.position.set(
        center.x+span*1.7,
        center.y+span*1.1,
        center.z+span*1.7
    );

    camera.near=span/100;
    camera.far=span*100;
    camera.updateProjectionMatrix();

    controls.update();
}


function animate(now){

    requestAnimationFrame(animate);

    if(playing && frames.length){

        const frameTime=1000/fps;

        if(now-lastTime>=frameTime){

            lastTime=now;

            current++;

            if(current>=frames.length)current=0;

            updateFrame(current);
        }
    }

    controls.update();
    renderer.render(scene,camera);
}

requestAnimationFrame(animate);


document.getElementById("play").onclick=()=>{

    playing=!playing;

    document.getElementById("play").textContent=
        playing ? "⏸ Pause" : "▶ Play";

    lastTime=performance.now();
};


document.getElementById("timeline").oninput=e=>{

    playing=false;
    document.getElementById("play").textContent="▶ Play";

    updateFrame(parseInt(e.target.value));
};


document.getElementById("generate").onclick=async()=>{

    const result=document.getElementById("result");

    result.textContent="Generating...";

    try{

        const response=await fetch("/generate",{
            method:"POST",
            headers:{"Content-Type":"application/json"},
            body:JSON.stringify({
                prompt:document.getElementById("prompt").value,
                duration:parseFloat(document.getElementById("duration").value)
            })
        });

        const data=await response.json();

        if(!response.ok)throw new Error(data.error || "Generate failed");

        frames=data.joints;
        fps=data.fps;
        current=0;

        edges=buildMST(frames[0]);

        createSkeleton(frames[0].length);

        const timeline=document.getElementById("timeline");
        timeline.max=frames.length-1;
        timeline.value=0;

        updateFrame(0);
        fitCamera();

        playing=true;
        document.getElementById("play").textContent="⏸ Pause";
        lastTime=performance.now();

        result.innerHTML=
            `Prompt: ${data.prompt}\n`+
            `Frames: ${data.frames}\n`+
            `FPS: ${data.fps}\n`+
            `Generation: ${data.seconds.toFixed(3)} s\n`+
            `RTF: ${data.rtf.toFixed(3)}\n`+
            `Shape: ${data.shape.join(" × ")}\n\n`+
            `<a href="${data.download}">Download NPZ</a>`;

    }catch(e){

        result.textContent="ERROR: "+e;
    }
};


window.addEventListener("resize",()=>{

    camera.aspect=viewer.clientWidth/viewer.clientHeight;
    camera.updateProjectionMatrix();

    renderer.setSize(viewer.clientWidth,viewer.clientHeight);
});

</script>
</main>
</body>
</html>
"""


class Handler(BaseHTTPRequestHandler):

    def log_message(self,fmt,*args):
        print("[WEB]",fmt%args)

    def send_data(self,data,ctype="text/html; charset=utf-8",status=200):
        self.send_response(status)
        self.send_header("Content-Type",ctype)
        self.send_header("Content-Length",str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):

        if self.path=="/":
            return self.send_data(HTML.encode())

        if self.path=="/health":
            return self.send_data(b'{"status":"ok"}',"application/json")

        if self.path.startswith("/download/"):

            name=Path(urllib.parse.unquote(self.path.split("/download/",1)[1])).name
            file=OUT_DIR/name

            if not file.exists():
                return self.send_data(b"Not found","text/plain",404)

            return self.send_data(file.read_bytes(),"application/octet-stream")

        return self.send_data(b"Not found","text/plain",404)


    def do_POST(self):

        if self.path!="/generate":
            return self.send_data(b'{"error":"not found"}',"application/json",404)

        try:

            n=int(self.headers.get("Content-Length","0"))
            req=json.loads(self.rfile.read(n))

            prompt=str(req.get("prompt","")).strip()
            duration=float(req.get("duration",2.0))

            if not prompt:
                raise ValueError("Prompt cannot be empty")

            duration=max(0.5,min(duration,10.0))

            fps=float(model.motion_rep.fps)
            num_frames=max(1,int(duration*fps))

            pad_mask=length_to_mask(torch.tensor([num_frames],device="cuda:0"))
            heading=torch.zeros(1,device="cuda:0")

            with gpu_lock:

                torch.cuda.synchronize(0)
                t0=time.perf_counter()

                with torch.inference_mode():

                    motion=model(
                        [prompt],
                        num_frames,
                        num_denoising_steps=int(model.diffusion.num_base_steps),
                        pad_mask=pad_mask,
                        first_heading_angle=heading,
                        motion_mask=None,
                        observed_motion=None,
                        cfg_weight=(2.0,2.0),
                        crop_history_length=model.num_frames_per_token,
                    )

                    output=model.motion_rep.inverse(
                        motion,
                        is_normalized=True
                    )

                torch.cuda.synchronize(0)
                dt=time.perf_counter()-t0

            joints=output["posed_joints"].detach().float().cpu().numpy()

            filename=f"ardy_{uuid.uuid4().hex[:10]}.npz"

            np.savez(
                OUT_DIR/filename,
                posed_joints=joints,
                fps=fps,
                prompt=prompt
            )

            response={
                "prompt":prompt,
                "frames":num_frames,
                "fps":fps,
                "seconds":dt,
                "rtf":dt/duration,
                "shape":list(joints.shape),
                "joints":joints[0].tolist(),
                "download":f"/download/{filename}"
            }

            return self.send_data(
                json.dumps(response).encode(),
                "application/json"
            )

        except Exception as e:

            print("Generate error:",repr(e))

            return self.send_data(
                json.dumps({"error":str(e)}).encode(),
                "application/json",
                500
            )


if "web_server" in globals():
    try: web_server.shutdown()
    except: pass

web_server=ThreadingHTTPServer((HOST,PORT),Handler)
web_thread=threading.Thread(target=web_server.serve_forever,daemon=True)
web_thread.start()

print("ARDY 3D Player ready")
print("http://127.0.0.1:2333")

ARDY 3D Player ready
http://127.0.0.1:2333
[WEB] "GET / HTTP/1.1" 200 -


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

[WEB] "POST /generate HTTP/1.1" 200 -


In [8]:
# Cell 8 — Cloudflare Tunnel
from pathlib import Path
import subprocess, urllib.request, re, time

CF = Path("/kaggle/working/cloudflared")

if not CF.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    CF.chmod(0o755)

tunnel_proc = subprocess.Popen(
    [str(CF), "tunnel", "--url", "http://127.0.0.1:2333", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url = None
deadline = time.time() + 30

while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line: continue
    print(line.rstrip())
    m = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n打开这个地址：", url)

2026-09-02T19:39:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-02T19:39:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-02T19:39:09Z INF +--------------------------------------------------------------------------------------------+
2026-09-02T19:39:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-02T19:39:09Z INF |  https://parliamentary-bedding-role-discusses.trycloud

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

[WEB] "POST /generate HTTP/1.1" 200 -
[WEB] "GET /download/ardy_96eb6563fd.npz HTTP/1.1" 200 -


## Wheel 建议

ARDY 真正需要本地编译的是 `motion_correction._motion_correction`，属于 **CMake + C++17 + pybind11/Eigen** 的 CPU native extension，不是 CUDA extension。

所以 wheel 主要绑定 Linux/x86_64 + Python ABI，不需要做 T4/CUDA 专属 wheel。

建议：
- patched ARDY wheel → GitHub Release
- 模型权重 → Hugging Face / Kaggle Dataset
- HF Token → Kaggle Secrets
- TensorRT `.trt` engine → 不建议作为通用 GitHub artifact

In [ ]:
# Cell 9 — 可选：构建 patched ARDY wheel
from pathlib import Path
import hashlib, subprocess, sys

WHEEL_OUT = Path("/kaggle/working/ardy_wheels")
WHEEL_OUT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-w", str(WHEEL_OUT)],
    cwd=str(REPO),
    check=True,
)

wheels = sorted(WHEEL_OUT.glob("ardy-*.whl"), key=lambda p:p.stat().st_mtime, reverse=True)
assert wheels
wheel = wheels[0]
print("Wheel:", wheel)
print("SHA256:", hashlib.sha256(wheel.read_bytes()).hexdigest())

## 执行顺序

第一次：运行 `Cell 0`，Kernel 会自动重启一次。

重启后：

`0 → 1 → 2 → 3 → 4 → 5 → 6`

需要 Demo：

`7 → 8`

需要构建 wheel：

最后运行 `Cell 9`。